In [ ]:
!pip install autoawq

In [ ]:
!pip install gptqmodel

In [ ]:
! pip install torch==2.0.1
! pip install torchvision==0.15.2
! pip install transformers==4.37.2
! pip install tiktoken==0.6.0
! pip install verovio==4.3.1
! pip install accelerate==0.28.0


### Prototipe Proses OCR Menggunakan Model GOT (OCR-2.0)
Pada prototipe ini, model OCR digunakan untuk mengekstrak gambar KTP.

In [2]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = 'ucaslcl/GOT-OCR2_0'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    device_map='auto',
    use_safetensors=True,
    pad_token_id=tokenizer.eos_token_id
)
model = model.eval().cuda()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/ucaslcl/GOT-OCR2_0:
- render_tools.py
- got_vision_b.py
. Make sure to double-check they do not contain any added malic

In [3]:
image_file = 'watermark.png'

result = model.chat(tokenizer, image_file, ocr_type='ocr')
print(result)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


PROVINSI DKI JAKARTA 
JAKARTA TIMUR 
NIK 
: 3175070101909999 
Nama 
: BILLY BUMBLEBEE SIFULAN 
Tempat/Tgl Lahir 
: SURABAYA, 01-01-1990 
Jenis Kelamin 
: LAKI-LAKI 
Alamat 
: JL DIMANA NO 100 
RT/RW 
: 001/001 
Kel/Desa 
: ANTAH BERANTAH 
Kecamatan 
: DUREN SAWIT 
Agama 
: ISLAM 
Status Perkawinan: KAWIN 
Pekerjaan 
: KARYAWAN SWASTA 
Kewarganegaraan : WNI 
Berlaku Hingga 
: SEUMUR HIDUP 
JAKARTA TIMUR 
01-01-2020 
21-10-2021 VERIFIKASIE-WALLET


### Prototipe Penggunaan LLM untuk Generate Data yang Lebih Terstruktur dari Hadil OCR

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AwqConfig


# Load model LLM

model_id = "Qwen/Qwen2.5-7B-Instruct-AWQ"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
)

quantization_config = AwqConfig(
    bits=4,
    fuse_max_seq_len=4096,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)

In [ ]:
# System prompt
system_prompt = """
Anda adalah AI Document Information Extractor yang bertugas
mengekstraksi informasi kandidat magang secara terstruktur dari
teks hasil OCR berbagai dokumen.

TUGAS UTAMA:
1. Mengidentifikasi jenis setiap dokumen.
2. Mengekstraksi informasi kandidat dari seluruh dokumen.
3. Menggabungkan informasi dari dokumen berbeda tanpa duplikasi.
4. Menilai kelengkapan setiap dokumen.
5. Menandai informasi yang ambigu, tidak lengkap, atau kemungkinan
   mengalami kesalahan OCR.
6. Menghasilkan output JSON yang valid sesuai schema yang diberikan.

ATURAN EKSTRAKSI:

1. Hanya gunakan informasi yang terdapat pada INPUT DATA OCR.
   Jangan mengarang, menebak, atau menambahkan informasi yang tidak
   terdapat dalam OCR.

2. Jika suatu informasi tidak ditemukan:
   - value = null
   - confidence = 0.0
   - ambiguous = false
   - evidence = null

3. Jika informasi ditemukan tetapi terdapat ketidakpastian karena
   kualitas OCR, pertahankan hasil OCR terbaik dan:
   - ambiguous = true

4. Kesalahan OCR boleh dikoreksi hanya jika koreksinya sangat kuat
   berdasarkan konteks kalimat, format dokumen, pola data, atau
   konsistensi dengan informasi lain.

5. Jangan melakukan koreksi spekulatif terhadap informasi identitas
   penting, khususnya:
   - NIK
   - nomor dokumen
   - nomor identitas lainnya

   Jika terdapat karakter yang tidak dapat dipastikan, pertahankan
   hasil OCR dan tandai ambiguous = true.

6. Jangan menyimpulkan informasi yang tidak tersedia.
   Contoh:
   Jangan menentukan tanggal lahir hanya berdasarkan umur.

7. Jika beberapa dokumen memiliki informasi yang sama, gunakan nilai
   yang memiliki bukti paling jelas dan paling kuat.

8. Jika terdapat konflik antar dokumen:
   - jangan memilih secara sembarangan
   - gunakan nilai yang paling didukung bukti
   - ambiguous = true

9. Setiap field hasil ekstraksi harus memiliki:
   - value
   - confidence
   - ambiguous
   - evidence

10. Confidence merupakan tingkat keyakinan terhadap hasil ekstraksi,
    bukan probabilitas kebenaran dokumen.
    Nilai harus berada pada rentang 0.0 sampai 1.0.

11. Evidence harus berisi potongan teks OCR yang menjadi dasar
    ekstraksi. Jangan berisi penjelasan atau asumsi.

12. Status kelengkapan dokumen:
    - complete   : seluruh informasi wajib tersedia
    - incomplete : terdapat informasi wajib yang hilang
    - uncertain  : informasi tersedia tetapi kualitas OCR membuat
                   kelengkapan tidak dapat dipastikan
    - unknown    : jenis dokumen tidak dapat diidentifikasi

13. Jangan menganggap dokumen lengkap hanya karena sebagian besar
    informasinya berhasil diekstraksi.

14. document_type harus berupa kategori dokumen yang paling sesuai.

15. Jika jenis dokumen tidak dapat diidentifikasi:
    document_type = "Unknown"

16. Output akhir HARUS berupa JSON valid.

17. JANGAN MEMBERIKAN:
    - markdown
    - ```json
    - komentar
    - penjelasan
    - teks tambahan

18. Semua key JSON harus mengikuti schema yang diberikan.
"""

# User prompt
ocr_text = """
PROVINSI DKI JAKARTA
JAKARTA TIMUR
NIK
: 3175070101909999
Nama
: BILLY BUMBLEBEE SIFULAN
Tempat/Tgl Lahir
: SURABAYA, 01-01-1990
Jenis Kelamin
: LAKI-LAKI
Alamat
: JL DIMANA NO 100
RT/RW
: 001/001
Kel/Desa
: ANTAH BERANTAH
Kecamatan
: DUREN SAWIT
Agama
: ISLAM
Status Perkawinan: KAWIN
Pekerjaan
: KARYAWAN SWASTA
Kewarganegaraan : WNI
Berlaku Hingga
: SEUMUR HIDUP
JAKARTA TIMUR
01-01-2020
21-10-2021 VERIFIKASIE-WALLET
"""


user_prompt = f"""
TUGAS:

Ekstrak informasi kandidat magang dari kumpulan teks hasil OCR
berikut.

Identifikasi jenis setiap dokumen, ekstrak informasi kandidat,
kemudian tentukan kelengkapan setiap dokumen.

OUTPUT JSON HARUS MENGIKUTI STRUKTUR BERIKUT:

{{
  "documents": {{
    "doc_1": {{
      "document_type": null,
      "document_number": {{
        "value": null,
        "confidence": 0.0,
        "ambiguous": false,
        "evidence": null
      }},
      "completeness_status": "unknown",
      "missing_fields": [],
      "confidence": 0.0
    }}
  }},

  "candidate_profile": {{
    "full_name": {{
      "value": null,
      "confidence": 0.0,
      "ambiguous": false,
      "evidence": null
    }},

    "nik": {{
      "value": null,
      "confidence": 0.0,
      "ambiguous": false,
      "evidence": null
    }},

    "birth_date": {{
      "value": null,
      "confidence": 0.0,
      "ambiguous": false,
      "evidence": null
    }},

    "address": {{
      "value": null,
      "confidence": 0.0,
      "ambiguous": false,
      "evidence": null
    }},

    "gender": {{
      "value": null,
      "confidence": 0.0,
      "ambiguous": false,
      "evidence": null
    }},

    "education": {{
      "level": null,
      "institution": null,
      "major": null,
      "graduation_year": null,
      "confidence": 0.0,
      "ambiguous": false,
      "evidence": null
    }}
  }},

  "document_completeness": {{
    "required_documents": [],
    "missing_documents": [],
    "complete": false
  }}
}}

ATURAN KHUSUS:

- document_type harus berupa kategori dokumen yang paling sesuai,
  misalnya:
  "KTP",
  "KK",
  "Ijazah",
  "Transkrip Nilai",
  "SKCK",
  "Hasil Medical Check Up",
  atau "Unknown".

- document_number digunakan untuk nomor dokumen seperti nomor KTP,
  nomor ijazah, atau nomor dokumen lainnya jika tersedia.

- birth_date harus menggunakan format DD-MM-YYYY jika tanggal, bulan,
  dan tahun dapat ditentukan dengan jelas.

- Jika format tanggal OCR ambigu dan tidak dapat dipastikan:
  gunakan null dan ambiguous = true.

- education.level berisi jenjang pendidikan terakhir yang dapat
  dibuktikan dari dokumen, misalnya:
  "SMA/SMK", "D3", "S1/D4", atau "S2".

- education.institution berisi nama institusi pendidikan jika tersedia.

- education.major berisi program studi atau jurusan jika tersedia.

- education.graduation_year berisi tahun kelulusan jika tersedia.

- address hanya boleh berisi alamat yang ditemukan dalam dokumen.
  Jangan membuat alamat berdasarkan informasi lain.

- Jika NIK atau nomor dokumen mengandung karakter yang tidak terbaca,
  jangan menebak angka yang hilang.

- Jika informasi yang sama terdapat pada beberapa dokumen, gabungkan
  informasi tersebut dan pilih nilai dengan bukti paling kuat.

- Jika terdapat konflik antar dokumen, tandai field tersebut:
  ambiguous = true

- Jika informasi tidak ditemukan:
  value = null
  confidence = 0.0
  ambiguous = false
  evidence = null

- missing_fields harus berisi nama informasi wajib yang tidak
  ditemukan pada dokumen tersebut.

INPUT DATA OCR:

{{
  "doc_1": {json.dumps(ocr_text, ensure_ascii=False)}
}}
"""

# Template chat
messages = [
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": user_prompt,
    },
]

text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer(
    [text_input],
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]

response_text = tokenizer.batch_decode(
    response_ids,
    skip_special_tokens=True,
)[0].strip()


print(response_text)

In [5]:
print(response_text)

```json
{
  "documents": {
    "doc_1": {
      "document_type": "KTP",
      "document_number": {
        "value": "3175070101909999",
        "confidence": 1.0,
        "ambiguous": false,
        "evidence": "NIK: 3175070101909999"
      },
      "completeness_status": "complete",
      "missing_fields": [],
      "confidence": 1.0
    }
  },

  "candidate_profile": {
    "full_name": {
      "value": "BILLY BUMBLEBEE SIFULAN",
      "confidence": 1.0,
      "ambiguous": false,
      "evidence": "Nama: BILLY BUMBLEBEE SIFULAN"
    },

    "nik": {
      "value": "3175070101909999",
      "confidence": 1.0,
      "ambiguous": false,
      "evidence": "NIK: 3175070101909999"
    },

    "birth_date": {
      "value": "01-01-1990",
      "confidence": 1.0,
      "ambiguous": false,
      "evidence": "Tempat/Tgl Lahir: SURABAYA, 01-01-1990"
    },

    "address": {
      "value": "JL DIMANA NO 100 RT/RW: 001/001 Kel/Desa: ANTAH BERANTAH Kecamatan: DUREN SAWIT",
      "confidence": 1.0,


In [7]:
# Validasi apakah response LLM JSON valid atau tidak
response_text = response_text.strip()

if response_text.startswith("```json"):
    response_text = response_text[len("```json"):].strip()

if response_text.endswith("```"):
    response_text = response_text[:-3].strip()

try:
    result = json.loads(response_text)

    print("\nJSON valid.")
    print(json.dumps(
        result,
        indent=1,
        ensure_ascii=False,
    ))

except json.JSONDecodeError as e:
    print("\nOutput bukan JSON valid.")
    print("Error:", e)


JSON valid.
{
 "documents": {
  "doc_1": {
   "document_type": "KTP",
   "document_number": {
    "value": "3175070101909999",
    "confidence": 1.0,
    "ambiguous": false,
    "evidence": "NIK: 3175070101909999"
   },
   "completeness_status": "complete",
   "missing_fields": [],
   "confidence": 1.0
  }
 },
 "candidate_profile": {
  "full_name": {
   "value": "BILLY BUMBLEBEE SIFULAN",
   "confidence": 1.0,
   "ambiguous": false,
   "evidence": "Nama: BILLY BUMBLEBEE SIFULAN"
  },
  "nik": {
   "value": "3175070101909999",
   "confidence": 1.0,
   "ambiguous": false,
   "evidence": "NIK: 3175070101909999"
  },
  "birth_date": {
   "value": "01-01-1990",
   "confidence": 1.0,
   "ambiguous": false,
   "evidence": "Tempat/Tgl Lahir: SURABAYA, 01-01-1990"
  },
  "address": {
   "value": "JL DIMANA NO 100 RT/RW: 001/001 Kel/Desa: ANTAH BERANTAH Kecamatan: DUREN SAWIT",
   "confidence": 1.0,
   "ambiguous": false,
   "evidence": "Alamat: JL DIMANA NO 100 RT/RW: 001/001 Kel/Desa: ANTAH B